# 🩺 Diabetes Prediction — MLOps Assignment 1
**Dataset:** `diabetes_unclean.csv`

---

## 📦 Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print(' Libraries imported successfully!')

---
# PART 1: Preprocessing and Data Cleaning

In [ ]:
# Load the dataset
df = pd.read_csv('diabetes_unclean.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Step 1 — Drop 'ID' and 'No_Pation' Columns

In [ ]:
# Drop unnecessary identifier columns
cols_to_drop = [col for col in ['ID', 'No_Pation'] if col in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f'Dropped columns: {cols_to_drop}')
print('Remaining columns:', df.columns.tolist())

### Step 2 — Fix Misspelled Gender Values

In [ ]:
# Display unique values in Gender column
print('Unique Gender values BEFORE cleaning:')
print(df['Gender'].unique())

In [ ]:
# Standardize Gender: strip whitespace, uppercase, then map variants to F/M
df['Gender'] = df['Gender'].astype(str).str.strip().str.upper()

# Map common misspellings/variants to correct values
gender_map = {
    'FEMALE': 'F', 'MALE': 'M',
    'Fe': 'F',     'fe': 'F',
    'f':  'F',     'm': 'M',
    'FEMAL': 'F',  'MAL': 'M',
}
df['Gender'] = df['Gender'].replace(gender_map)

# Keep only valid values
df = df[df['Gender'].isin(['F', 'M'])]

print('Unique Gender values AFTER cleaning:')
print(df['Gender'].unique())
print('Value counts:')
print(df['Gender'].value_counts())

### Step 3 — One-Hot Encoding on Categorical Columns (Except Target)

In [ ]:
# Identify target column
target_col = 'CLASS'   # change if your CSV uses a different name

# Identify categorical columns (excluding target)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != target_col]
print('Categorical columns to encode:', cat_cols)

# One-hot encode
df = pd.get_dummies(df, columns=cat_cols, drop_first=False)
print(' One-hot encoding done. New shape:', df.shape)
df.head()

### Step 4 — Handle Missing Values

In [ ]:
# Check missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

In [ ]:
# Fill numeric columns with mean, categorical/bool with mode
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            df[col].fillna(df[col].mean(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

print(' Missing values after handling:', df.isnull().sum().sum())
print('Final dataset shape:', df.shape)

---
# PART 2: Exploratory Data Analysis

### Plot 1 — Bar Chart: Gender Distribution

In [ ]:
# Reconstruct gender from one-hot columns for plotting
gender_cols = [c for c in df.columns if c.startswith('Gender_')]
if gender_cols:
    gender_series = df[gender_cols].idxmax(axis=1).str.replace('Gender_', '')
else:
    gender_series = df['Gender']   # in case encoding wasn't done

gender_counts = gender_series.value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(gender_counts.index, gender_counts.values,
       color=['#4C72B0', '#DD8452'], edgecolor='black', width=0.4)
ax.set_title('Distribution of Male and Female Patients', fontsize=14, fontweight='bold')
ax.set_xlabel('Gender')
ax.set_ylabel('Count')
for i, v in enumerate(gender_counts.values):
    ax.text(i, v + 1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/plot1_gender_distribution.png', dpi=150)
plt.show()
print(' Plot saved.')

### Plot 2 — Histogram: Age Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['AGE'], bins=20, color='#4C72B0', edgecolor='black', alpha=0.85)
ax.set_title('Age Distribution of Patients', fontsize=14, fontweight='bold')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('screenshots/plot2_age_distribution.png', dpi=150)
plt.show()
print('Plot saved.')

### Plot 3 — Histogram: BMI Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['BMI'], bins=20, color='#DD8452', edgecolor='black', alpha=0.85)
ax.set_title('BMI Distribution of Patients', fontsize=14, fontweight='bold')
ax.set_xlabel('BMI')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('screenshots/plot3_bmi_distribution.png', dpi=150)
plt.show()
print('Plot saved.')

### Plot 4 — Scatter Plot: BMI vs HbA1c (color-coded by class)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    df['BMI'], df['HbA1c'],
    c=df[target_col].astype('category').cat.codes,
    cmap='Set1', alpha=0.6, edgecolors='k', linewidths=0.3
)
legend_labels = df[target_col].unique()
handles, _ = scatter.legend_elements()
ax.legend(handles, sorted(legend_labels), title='Diabetes Class')
ax.set_title('BMI vs HbA1c (color-coded by Diabetes Class)', fontsize=13, fontweight='bold')
ax.set_xlabel('BMI')
ax.set_ylabel('HbA1c')
plt.tight_layout()
plt.savefig('screenshots/plot4_bmi_vs_hba1c.png', dpi=150)
plt.show()
print('Plot saved.')

### Plot 5 — Scatter Plot: Age vs HbA1c (color-coded by class)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    df['AGE'], df['HbA1c'],
    c=df[target_col].astype('category').cat.codes,
    cmap='Set1', alpha=0.6, edgecolors='k', linewidths=0.3
)
handles, _ = scatter.legend_elements()
ax.legend(handles, sorted(legend_labels), title='Diabetes Class')
ax.set_title('Age vs HbA1c (color-coded by Diabetes Class)', fontsize=13, fontweight='bold')
ax.set_xlabel('Age (years)')
ax.set_ylabel('HbA1c')
plt.tight_layout()
plt.savefig('screenshots/plot5_age_vs_hba1c.png', dpi=150)
plt.show()
print(' Plot saved.')

### Plot 6 — Box Plot: BMI by Diabetes Class

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
groups = [df[df[target_col] == cls]['BMI'].values
          for cls in sorted(df[target_col].unique())]
bp = ax.boxplot(groups, patch_artist=True,
                labels=sorted(df[target_col].unique()))
colors = ['#4C72B0', '#DD8452', '#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title('BMI Distribution by Diabetes Class', fontsize=13, fontweight='bold')
ax.set_xlabel('Diabetes Class')
ax.set_ylabel('BMI')
plt.tight_layout()
plt.savefig('screenshots/plot6_bmi_boxplot.png', dpi=150)
plt.show()
print(' Plot saved.')

---
# PART 3: Model Training

### Train-Test Split (70% / 30%)

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Testing set size  : {X_test.shape[0]} samples')

### Train & Evaluate All 5 Models

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'SVM'                 : SVC(kernel='rbf', random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=5),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    trained_models[name] = model
    results.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'Recall'   : round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'F1-Score' : round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
    })
    print(f' {name} trained.')

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
print('\n===== MODEL COMPARISON TABLE =====')
results_df

---
# PART 4: Model Selection and Saving

In [ ]:
# Select best model based on F1-Score
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print(f' Best Model: {best_model_name}')
print(f'   F1-Score : {results_df.iloc[0]["F1-Score"]}')
print(f'   Accuracy : {results_df.iloc[0]["Accuracy"]}')

In [ ]:
# Save model
joblib.dump(best_model, 'diabetes_model.pkl')
print('Model saved as diabetes_model.pkl')

# Save training column names
training_columns = X_train.columns.tolist()
joblib.dump(training_columns, 'training_columns.pkl')
print(' Training columns saved as training_columns.pkl')
print('Columns:', training_columns)